<a href="https://colab.research.google.com/github/CSB-KaracaLab/DynaPIN/blob/main/DynaPIN_Colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DynaPIN in Colab

**Dynamic interface fingerprints from a molecular dynamics trajectory — quality control,
residue properties and interaction networks, in one command.**

A protein–protein interface is usually reported as a list of contacts taken from one
structure. Over a trajectory that list is not fixed: some contacts hold throughout, others
open and close, and residues move between core, rim and surface as the partner breathes
against them. [DynaPIN](https://github.com/CSB-KaracaLab/DynaPIN) measures that.

> Barlas AB, Özsan A, Prévost C, Sacquin-Mora S, Karaca E.
> *DynaPIN: A Tool for Characterizing Dynamic Protein Interfaces.*
> **J Mol Biol** 2026;438:169942 ·
> [doi:10.1016/j.jmb.2026.169942](https://doi.org/10.1016/j.jmb.2026.169942)

### Why a notebook

DynaPIN normally installs through conda, because it needs DSSP, DockQ and a compiled
interface-analysis extension alongside the Python packages. That is the main thing standing
between a curious reader and a first run. Here it is all done for you in one cell.

### The worked example

**2A9K**, a two-chain complex, over 21 frames of molecular dynamics from the
[DynaBench](http://www-lbt.ibpc.fr/DynaBench) set. Small enough to analyse in a couple of
minutes, and real: the same trajectory the DynaPIN logo is drawn from.

> Barlas AB, Laurent B, Karaca E, Prévost C, Sacquin-Mora S.
> *DynaBench: a benchmark of protein–protein complexes in motion.*
> **J Mol Biol** 2026;438:169942 ·
> [doi:10.1016/j.jmb.2026.169942](https://doi.org/10.1016/j.jmb.2026.169942)

Swap in your own trajectory at step 2 — any format MDAnalysis reads, with a topology that
carries chain identifiers.

In [ ]:
#@title 1. Install DynaPIN { display-mode: "form" }
#@markdown Installs DSSP, the Python dependencies, the vendored `interfacea` extension and
#@markdown DynaPIN itself. Three to five minutes, once per session.
import os, subprocess, sys, time

t0 = time.time()

if not os.path.isdir("DynaPIN"):
    subprocess.run(["git", "clone", "--quiet", "--depth", "1",
                    "https://github.com/CSB-KaracaLab/DynaPIN.git"], check=True)
REPO = os.path.abspath("DynaPIN")

if not os.path.isfile("DYNAPIN_READY"):
    # mkdssp is a binary, not a Python package; DynaPIN shells out to it for the
    # secondary-structure analysis
    print("installing DSSP ...", flush=True)
    os.system("apt-get -qq install -y dssp > /dev/null 2>&1")

    print("installing python dependencies ...", flush=True)
    os.system("pip install -q MDAnalysis freesasa matplotlib numpy openmm pandas "
              "seaborn pdb-tools DockQ py3Dmol 2>&1 | tail -3")

    # interfacea is not on PyPI; the copy DynaPIN uses is vendored in the repository and
    # builds a C extension, so it is installed from source
    print("building interfacea ...", flush=True)
    os.system(f"pip install -q -e {REPO}/interfacea_src 2>&1 | tail -3")

    print("installing DynaPIN ...", flush=True)
    os.system(f"pip install -q {REPO} 2>&1 | tail -3")
    os.system("touch DYNAPIN_READY")

import shutil  # noqa: E402

checks = {
    "dynapin": shutil.which("dynapin"),
    "mkdssp": shutil.which("mkdssp") or shutil.which("dssp"),
    "DockQ": shutil.which("DockQ"),
    "pdb_splitmodel": shutil.which("pdb_splitmodel"),
}
for name, path in checks.items():
    print(f"  {name:<16} {'found' if path else 'MISSING'}")

import DynaPIN  # noqa: E402
print(f"\nDynaPIN imported, ready in {time.time() - t0:.0f} s")
if not checks["mkdssp"]:
    print("\nDSSP is missing: the secondary-structure part of the residue analysis will "
          "fail. Everything else still runs.")

In [ ]:
#@title 2. The trajectory to analyse { display-mode: "form" }
#@markdown The default is 2A9K from DynaBench: a two-chain complex over 21 frames, about
#@markdown 1.2 MB in all. Untick to upload your own instead — any trajectory MDAnalysis
#@markdown reads, plus a topology whose PDB carries **chain identifiers**, which DynaPIN
#@markdown needs to tell the two sides of the interface apart.
use_the_example = True  #@param {type:"boolean"}
chains = "A,B"  #@param {type:"string"}
stride = 1  #@param {type:"integer"}
import collections, urllib.request
from pathlib import Path

RAW = ("https://raw.githubusercontent.com/CSB-KaracaLab/bioemu2-colab/main/md_reference/")

if use_the_example:
    TOPOLOGY, TRAJECTORY = Path("2A9K_md_topology.pdb"), Path("2A9K_md_dynabench.xtc")
    for path in (TOPOLOGY, TRAJECTORY):
        if not path.exists():
            urllib.request.urlretrieve(RAW + path.name, path)
else:
    from google.colab import files
    print("upload the topology (.pdb/.psf) and then the trajectory (.xtc/.dcd/.trr/.pdb)")
    uploaded = files.upload()
    names = list(uploaded)
    TOPOLOGY = Path(next(n for n in names if n.endswith((".pdb", ".psf"))))
    TRAJECTORY = Path(next(n for n in names if n != TOPOLOGY.name))

present = collections.Counter()
residues = set()
for line in TOPOLOGY.read_text(errors="ignore").splitlines():
    if line.startswith(("ATOM", "HETATM")):
        present[line[21]] += 1
        residues.add((line[21], line[22:27]))

print(f"topology   : {TOPOLOGY}  ({TOPOLOGY.stat().st_size / 1e6:.1f} MB)")
print(f"trajectory : {TRAJECTORY}  ({TRAJECTORY.stat().st_size / 1e6:.1f} MB)")
print(f"chains in the topology: {dict(present)}")
print(f"residues: {len(residues)}")

wanted = [c.strip() for c in chains.split(",") if c.strip()]
missing = [c for c in wanted if c not in present]
if missing:
    print(f"\nChain(s) {missing} are not in the topology. DynaPIN needs two chain IDs that "
          f"exist; pick from {sorted(present)}.")
elif len(wanted) != 2:
    print(f"\nDynaPIN analyses one interface at a time: give exactly two chains, "
          f"e.g. '{sorted(present)[0]},{sorted(present)[1]}'.")
else:
    print(f"\ninterface to analyse: chain {wanted[0]} against chain {wanted[1]}")

In [ ]:
#@title 3. Run the three analysis modules { display-mode: "form" }
#@markdown Quality control, the residue-based analysis and the interaction network, plus
#@markdown every plot. Output is streamed as DynaPIN produces it.
#@markdown
#@markdown FoldX is left out: its licence is academic and it has to be downloaded by hand,
#@markdown so the per-residue energy decomposition is skipped. Everything else runs.
import subprocess, time
from pathlib import Path

OUTPUT = Path("DynaPIN_run")

command = ["dynapin",
           f"--output_dir={OUTPUT}",
           f"--trajectory_file={TRAJECTORY}",
           f"--topology_file={TOPOLOGY}",
           f"--chains={chains}",
           f"--stride={stride}",
           "--commands=all_analysis,all_plots"]
print(" ".join(command), "\n")

t0 = time.time()
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    line = line.rstrip()
    if line:
        print("  " + line[:160], flush=True)
process.wait()
print(f"\nexit code {process.returncode}, {time.time() - t0:.0f} s")

tables = sorted((OUTPUT / "tables").glob("*.csv")) if (OUTPUT / "tables").is_dir() else []
figures = sorted((OUTPUT / "figures").glob("*.png")) if (OUTPUT / "figures").is_dir() else []
print(f"\n{len(tables)} tables, {len(figures)} figures")
for t in tables:
    print(f"  tables/{t.name}")

In [ ]:
#@title 4. Was the simulation worth analysing? { display-mode: "form" }
#@markdown DynaPIN's quality control: how far the complex drifts, how compact it stays, and
#@markdown the CAPRI metrics of every frame measured against the first one. DockQ falling
#@markdown away from 1.0 is the interface changing, not the simulation going wrong.
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

INK, INK_SOFT, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"

qc = pd.read_csv(OUTPUT / "tables/qc_trajectory_metrics.csv")
print(f"{len(qc)} frames, columns: {', '.join(qc.columns)}\n")

has_dockq = "DockQ" in qc.columns
panels = [("Complex RMSD", "RMSD (Å)", BLUE), ("Complex RG", "radius of gyration (Å)", GREEN)]
if has_dockq:
    panels.append(("DockQ", "DockQ vs frame 0", ORANGE))

fig, axes = plt.subplots(1, len(panels), figsize=(3.5 * len(panels), 2.9), dpi=140)
for ax, (column, label, colour) in zip(axes, panels):
    ax.plot(qc["Frame"], qc[column], lw=2, color=colour)
    ax.set_xlabel("frame", fontsize=9, color=INK_SOFT)
    ax.set_ylabel(label, fontsize=9, color=INK_SOFT)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_SOFT, length=0, labelsize=8.5)
fig.suptitle("Quality control", fontsize=10.5, color=INK, x=0.01, ha="left")
plt.tight_layout()
plt.show()

if has_dockq:
    print(f"DockQ over the trajectory: {qc.DockQ.min():.3f} to {qc.DockQ.max():.3f}")
    print(f"interface RMSD (iRMSD)   : up to {qc.iRMSD.max():.2f} Å")
    print(f"native contacts kept     : {qc.fnat.min():.2f} at the least")
else:
    print("No CAPRI metrics in this table. DynaPIN computes them only when it finds more "
          "than one chain, and they need the per-frame models in models/ - if that folder "
          "came out empty, pdb_splitmodel could not be found.")

In [ ]:
#@title 5. The dynamic interface fingerprint { display-mode: "form" }
#@markdown Each interaction DynaPIN found, and the share of frames it exists in. The point
#@markdown of the tool is this column: a contact present in every frame is a different
#@markdown thing from one present in a fifth of them, and a single structure cannot tell
#@markdown them apart.
import numpy as np

pairs = pd.read_csv(OUTPUT / "tables/int_pairwise_trajectory.csv")
n_frames = pairs["Frame"].nunique()
print(f"{len(pairs)} interactions recorded over {n_frames} frames")

# one row per residue pair and interaction type, with the fraction of frames it survives
pairs["pair"] = (pairs.chain_a + ":" + pairs.residue_a + " — "
                 + pairs.chain_b + ":" + pairs.residue_b)
persistence = (pairs.groupby(["pair", "itype"])["Frame"].nunique() / n_frames
               ).reset_index(name="persistence")

TYPE_COLOUR = {"hbond": BLUE, "hydrophobic": ORANGE, "ionic": GREEN}
kinds = [k for k in TYPE_COLOUR if k in set(persistence.itype)]
print("interaction types:", ", ".join(kinds))

top = persistence.sort_values("persistence", ascending=False).head(25).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.4, 0.26 * len(top) + 1.4), dpi=140)
ax.barh(range(len(top)), top.persistence * 100,
        color=[TYPE_COLOUR.get(t, INK_SOFT) for t in top.itype], height=0.72)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top.pair, fontsize=7.5)
ax.set_xlabel("share of frames the interaction exists in (%)", fontsize=9, color=INK_SOFT)
ax.set_xlim(0, 100)
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=INK_SOFT, length=0, labelsize=8.5)
handles = [plt.Line2D([], [], marker="s", ls="", markersize=7, color=TYPE_COLOUR[k],
                      label=k) for k in kinds]
ax.legend(handles=handles, frameon=False, fontsize=8.5, labelcolor=INK_SOFT,
          loc="lower right")
ax.set_title("The 25 most persistent interface interactions", fontsize=10.5, color=INK,
             pad=10, loc="left")
plt.tight_layout()
plt.show()

always = persistence[persistence.persistence == 1.0]
sometimes = persistence[(persistence.persistence > 0) & (persistence.persistence < 1.0)]
print(f"\n{len(always)} interactions hold in every frame")
print(f"{len(sometimes)} come and go")
print(f"{len(persistence)} distinct interactions in total — a single structure would have "
      f"reported roughly {len(pairs[pairs.Frame == pairs.Frame.min()])}.")

In [ ]:
#@title 6. Which residues actually move? { display-mode: "form" }
#@markdown Per-residue fluctuation, with the interface residues marked. A rigid core and a
#@markdown mobile interface is the pattern DynaPIN was built to pick out.
rmsf = pd.read_csv(OUTPUT / "tables/qc_residue_rmsf.csv")
interface_residues = {(r.chain_a, r.residue_a) for r in pairs.itertuples()} | \
                     {(r.chain_b, r.residue_b) for r in pairs.itertuples()}
# the RMSF table numbers residues; the interaction table names them, e.g. GLY71
by_number = {(chain, int("".join(ch for ch in name if ch.isdigit())))
             for chain, name in interface_residues
             if any(ch.isdigit() for ch in name)}

fig, axes = plt.subplots(len(rmsf.Molecule.unique()), 1,
                         figsize=(7.6, 2.0 * rmsf.Molecule.nunique() + 0.6), dpi=140,
                         sharex=False)
axes = np.atleast_1d(axes)
for ax, (chain, block) in zip(axes, rmsf.groupby("Molecule")):
    at_interface = block["Residue Number"].isin(
        [n for c, n in by_number if c == chain])
    ax.plot(block["Residue Number"], block["RMSF"], lw=1.4, color=INK_SOFT, zorder=2)
    ax.scatter(block["Residue Number"][at_interface], block["RMSF"][at_interface],
               s=16, color=ORANGE, zorder=3, label="at the interface")
    ax.set_ylabel(f"chain {chain}\nRMSF (Å)", fontsize=9, color=INK_SOFT)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_SOFT, length=0, labelsize=8.5)
    ax.legend(frameon=False, fontsize=8.5, labelcolor=INK_SOFT, loc="upper right")
axes[-1].set_xlabel("residue number", fontsize=9, color=INK_SOFT)
axes[0].set_title("Per-residue fluctuation", fontsize=10.5, color=INK, pad=10, loc="left")
plt.tight_layout()
plt.show()

for chain, block in rmsf.groupby("Molecule"):
    at_interface = block["Residue Number"].isin([n for c, n in by_number if c == chain])
    if at_interface.any():
        print(f"chain {chain}: interface residues move "
              f"{block.RMSF[at_interface].mean():.2f} Å on average, "
              f"the rest {block.RMSF[~at_interface].mean():.2f} Å")

In [ ]:
#@title 7. The plots DynaPIN drew itself { display-mode: "form" }
#@markdown Everything above was read back out of the tables. DynaPIN also writes its own
#@markdown figures, which is what you get from a plain command-line run.
from IPython.display import Image, display

figures = sorted((OUTPUT / "figures").glob("*.png"))
print(f"{len(figures)} figures in {OUTPUT}/figures\n")
for path in figures:
    print(path.name)
    display(Image(filename=str(path), width=560))

In [ ]:
#@title 8. Take the results with you { display-mode: "form" }
#@markdown The whole output directory as one zip: tables, figures, the per-frame models
#@markdown and the parameters each step ran with.
download_now = True  #@param {type:"boolean"}
import shutil

archive = shutil.make_archive(f"{OUTPUT.name}_results", "zip", OUTPUT)
print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.1f} MB)")
for item in sorted(OUTPUT.iterdir()):
    print(f"  {item.name}")

if download_now:
    from google.colab import files
    files.download(archive)

## Running this on your own trajectory

Replace the example at step 2 and the rest follows:

- **Trajectory**: `.dcd`, `.xtc`, `.trr` or a multi-model `.pdb`.
- **Topology**: needed for everything except a multi-model PDB, and **it must carry chain
  identifiers** — DynaPIN uses them to tell the two sides of the interface apart.
- **Chains**: exactly two, e.g. `A,B`. For a complex with more, pick the pair you care about.
- **Stride**: read every n-th frame. The quickest way to make a first pass cheap on a long
  trajectory.

### What is missing here

**FoldX.** The per-residue energy decomposition — van der Waals and electrostatic
contributions — needs the FoldX binary, whose licence is academic and which has to be
[downloaded by hand](https://foldxsuite.crg.eu/). Pass `--foldx_path /path/to/foldx` to a
local run and that module switches on. Everything else in this notebook runs without it.

### The three test cases from the paper

The trajectories and the full outputs for the rigid, medium and difficult cases are on
Zenodo at [10.5281/zenodo.20180638](https://doi.org/10.5281/zenodo.20180638) — about
2.7 GB in two archives, so they are not downloaded here, but they are the reference if you
want to check DynaPIN reproduces what the paper reports.

### Citing

> Barlas AB, Özsan A, Prévost C, Sacquin-Mora S, Karaca E. *DynaPIN: A Tool for
> Characterizing Dynamic Protein Interfaces.* **J Mol Biol** 2026;438:169942 ·
> [doi:10.1016/j.jmb.2026.169942](https://doi.org/10.1016/j.jmb.2026.169942)

DynaPIN stands on MDAnalysis, pdb-tools, DockQ, FreeSASA, DSSP, FoldX and interfacea; the
repository's README lists each with its own citation, and they deserve it.